# Matplotlib: turning numbers into pictures

Two cities have a week of temperature observations. A reader wants to know when they differ and whether one is consistently warmer. Choose a chart for each comparison, inspect how summaries hide observations, and finish with a report whose units and missing readings are visible.

A table can tell us the temperature on each day. A chart makes a warm spell, an unusual reading, or a difference between two places easier to see. Matplotlib gives us control over how those numbers become a picture.

We will move from one line chart to comparisons, distributions, and a small report. The temperature and rainfall values below are **invented teaching data**, not weather observations. Every example runs offline after installation.

Run the cells from top to bottom. Each section explains the question before drawing the chart.

## 1. Install and import

If needed, run `%pip install numpy matplotlib` in a separate notebook cell. NumPy holds our measurements; Matplotlib draws them. We use `plt.subplots()` to create a figure and an explicit plotting area.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("Matplotlib:", matplotlib.__version__)
days = np.arange(1, 8)
city_a = np.array([18, 20, 21, 19, 22, 24, 23])
city_b = np.array([16, 17, 19, 18, 20, 21, 20])
rainfall = np.array([4, 0, 0, 8, 2, 0, 1])

## 2. Figure, Axes, and Axis

Think of a Figure as the whole page and an Axes as one chart on that page. An Axis controls a coordinate direction, including its ticks and labels.

| Object | Responsibility | Example |
| --- | --- | --- |
| Figure (`fig`) | Whole canvas, layout, export | `fig.savefig(...)` |
| Axes (`ax`) | Data marks, title, labels, legend | `ax.plot(...)` |
| Axis | Ticks and scale along one direction | `ax.xaxis` |

The plural-looking name `Axes` still means one plotting area. The following chart connects readings in day order; that connection is meaningful because time is ordered.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.plot(days, city_a, marker="o")
ax.set(title="City A: one week of temperatures", xlabel="Day", ylabel="Temperature (°C)")
plt.show()

## 3. Compare cities on the same dates and scale

Predict which city should be higher on each day. Confirm that the lines use the same units and corresponding x positions. A visual difference is meaningful only when observations are aligned.

Both lines share the same units and day positions, so one Axes supports a direct comparison. A `label` names each series, and `legend()` displays those names. Markers and line styles make the series distinguishable without relying only on color.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.plot(days, city_a, color="tab:blue", marker="o", label="City A")
ax.plot(days, city_b, color="tab:orange", marker="s", linestyle="--", label="City B")
ax.set(title="Compare temperatures on the same days", xlabel="Day", ylabel="Temperature (°C)")
ax.set_xticks(days)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

## 4. Match the chart to the question

| Question | Chart | Method |
| --- | --- | --- |
| How does a value change over ordered time? | Line | `plot` |
| How do named categories compare? | Bar | `bar` or `barh` |
| Do two measurements vary together? | Scatter | `scatter` |
| How are numerical values distributed? | Histogram | `hist` |
| How do distributions compare? | Box plot | `boxplot` |
| Where are high and low values in a matrix? | Heatmap | `imshow` |

A line between unrelated categories can imply an order that the data does not have. Choose the chart from the question before changing colors or fonts.

## 5. Bars compare categories

Compare the mean temperature of the two cities. Each bar represents a calculated summary, not one day. A zero baseline makes bar lengths proportional to the displayed values; a narrow cropped baseline would exaggerate the difference.

In [ ]:
means = [city_a.mean(), city_b.mean()]
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
bars = ax.bar(["City A", "City B"], means, color=["tab:blue", "tab:orange"])
ax.bar_label(bars, fmt="%.1f", padding=3)
ax.set(title="Average temperature across seven days", ylabel="Mean temperature (°C)", ylim=(0, 26))
plt.show()

## 6. Scatter plots preserve paired observations

Each point below represents the same day in both cities. The horizontal coordinate is City A's reading; the vertical coordinate is City B's reading. The dashed reference is the line $y=x$: points below it indicate that City B was cooler.

The colorbar encodes day number. A pattern in seven invented pairs is an illustration, not evidence of a causal relationship.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
points = ax.scatter(city_a, city_b, c=days, cmap="viridis", s=85, edgecolors="black")
ax.plot([15, 25], [15, 25], color="gray", linestyle="--", label="Equal temperature")
ax.set(xlabel="City A temperature (°C)", ylabel="City B temperature (°C)", title="Paired daily readings", xlim=(15, 25), ylim=(15, 25))
ax.set_aspect("equal", adjustable="box")
fig.colorbar(points, ax=ax, label="Day", ticks=days)
ax.legend()
plt.show()

## 7. Check whether the histogram shape depends on your bins

Predict how fewer bins will change the apparent distribution, then compare several choices. Describe features that persist rather than assigning meaning to every bump in a small sample.

A histogram counts values within bins. Here we generate 200 synthetic readings using a fixed random seed, so rerunning this notebook gives the same sample. Unlike a bar chart of named categories, the horizontal axis is a numerical scale.

Try changing the number of bins from 12 to 5 or 30. The measurements remain the same, but the visible shape changes.

In [ ]:
rng = np.random.default_rng(42)
readings = rng.normal(loc=21, scale=2.5, size=200)
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
counts, edges, patches = ax.hist(readings, bins=12, edgecolor="white", color="tab:blue")
ax.axvline(readings.mean(), color="black", linestyle="--", label=f"Mean: {readings.mean():.1f} °C")
ax.set(title="Distribution of synthetic temperature readings", xlabel="Temperature (°C)", ylabel="Number of readings")
ax.legend()
plt.show()
print("Readings counted:", int(counts.sum()))

## 8. Box plots summarize spread

The box spans the 25th to 75th percentiles; its line marks the median. With the default rule, whiskers reach the most extreme observations within 1.5 interquartile ranges of the box. Points beyond the whiskers are displayed separately; they are not automatically errors.

Seven readings per city make a small sample. Overlaying the observations helps us see what the summary contains.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
ax.boxplot([city_a, city_b])
ax.set_xticks([1, 2], ["City A", "City B"])
for position, values in enumerate([city_a, city_b], start=1):
    ax.scatter(np.full(values.size, position), values, zorder=3, alpha=0.7)
ax.set(title="Spread across seven days", ylabel="Temperature (°C)")
plt.show()

## 9. Subplots separate different units

Temperature and rainfall use different units. Put them in separate panels with a shared day axis. `plt.subplots(2, 1)` returns two Axes arranged vertically; indexing selects the panel to edit.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True, layout="constrained")
axes[0].plot(days, city_a, marker="o", color="tab:blue")
axes[0].set(title="City A: temperature and rainfall", ylabel="Temperature (°C)")
axes[1].bar(days, rainfall, color="tab:purple")
axes[1].set(xlabel="Day", ylabel="Rainfall (mm)")
axes[1].set_xticks(days)
plt.show()

## 10. Heatmaps show a matrix

Rows identify cities and columns identify days. A shared color scale makes the cells comparable. Always label the colorbar with the measured quantity and unit. `imshow` returns the object used to create the colorbar.

In [ ]:
temperatures = np.vstack([city_a, city_b])
fig, ax = plt.subplots(figsize=(9, 3), layout="constrained")
heatmap = ax.imshow(temperatures, cmap="viridis", aspect="auto", vmin=15, vmax=25)
ax.set_xticks(np.arange(7), days)
ax.set_yticks([0, 1], ["City A", "City B"])
ax.set(title="Temperature by city and day", xlabel="Day")
fig.colorbar(heatmap, ax=ax, label="Temperature (°C)")
plt.show()

## 11. Annotations explain an observation

An annotation attaches a short explanation to a data coordinate. `xy` points to the reading; `xytext` positions the explanation. Calculate the peak location instead of hard-coding it, so the annotation follows the data.

In [ ]:
peak = int(np.argmax(city_a))
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.plot(days, city_a, marker="o")
ax.annotate(f"Peak: {city_a[peak]} °C", xy=(days[peak], city_a[peak]),
            xytext=(4, 25.5), arrowprops={"arrowstyle": "->"})
ax.set(title="Find the warmest day", xlabel="Day", ylabel="Temperature (°C)", ylim=(17, 27))
plt.show()

## 12. Show where a sensor failed to report

Before drawing, distinguish a missing reading from a zero-degree reading. Locate the gap in the output and explain why connecting across it would imply information you do not have.

A missing measurement is not zero. NumPy's `nan` represents the missing value here, and Matplotlib leaves a gap in the line. Removing that position entirely would connect across the gap and could suggest observations that were never recorded.

In [ ]:
with_gap = city_a.astype(float).copy()
with_gap[3] = np.nan
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.plot(days, with_gap, marker="o")
ax.set(title="Day 4 has no recorded reading", xlabel="Day", ylabel="Temperature (°C)")
ax.set_xticks(days)
plt.show()

## 13. Logarithmic scales show ratios

On a linear scale, equal distances represent equal differences. On a logarithmic scale, equal distances represent equal ratios. A doubling sequence therefore forms a straight line on a log-scaled vertical axis. This example uses positive values; zero and negative values cannot be shown on an ordinary logarithmic scale.

In [ ]:
steps = np.arange(8)
values = 2.0 ** steps
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax in axes:
    ax.plot(steps, values, marker="o")
    ax.set(xlabel="Step", ylabel="Value")
axes[0].set_title("Linear scale")
axes[1].set_yscale("log", base=2)
axes[1].set_title("Log scale: equal spacing for doubling")
plt.show()

## 14. Make the weekly comparison independently readable

Inspect the difference panel against the original two lines: its sign should agree with which city is warmer. Check the zero reference, units and legend, then open the saved report and verify that those explanations survive export.

Combine a temperature comparison with daily differences. The difference is $T_A-T_B$ in degrees Celsius: a positive value means City A was warmer. A horizontal zero line makes that meaning visible.

Use a local style context so the style applies only to this report. Save the Figure before showing it. PNG is a raster image; SVG preserves vector shapes for resizing. Running this cell creates an `outputs` folder and replaces these two report files if they already exist.

In [ ]:
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
with plt.style.context("default"):
    report, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True, layout="constrained")
    axes[0].plot(days, city_a, marker="o", label="City A")
    axes[0].plot(days, city_b, marker="s", linestyle="--", label="City B")
    axes[0].set(ylabel="Temperature (°C)", title="Weekly temperature report — teaching data")
    axes[0].legend()
    axes[1].bar(days, city_a - city_b, color="tab:green")
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set(xlabel="Day", ylabel="City A − City B (°C)")
    axes[1].set_xticks(days)
    for ax in axes:
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
    for extension in ["png", "svg"]:
        destination = output_dir / f"matplotlib_weekly_report.{extension}"
        report.savefig(destination, dpi=150, bbox_inches="tight")
        print("Saved:", destination)
    plt.show()

## 15. Practice and common mistakes

1. Add a third city's seven temperatures and update the comparison and legend.
2. Change the histogram bins. Explain which features persist across choices.
3. Annotate the largest city-to-city difference in the report.
4. Replace one reading with `np.nan` and use `np.nanmean` when calculating the mean of available readings.

| Symptom | What to check |
| --- | --- |
| x and y have different lengths | Each coordinate needs its matching value |
| Legend is empty | Give plotted series a `label`, then call `legend()` |
| Labels overlap | Use `layout="constrained"` and a larger figure if necessary |
| Saved plot is blank | Keep the intended Figure reference and save before `show()` |
| Too many figures remain open in a loop | Save each figure, then call `plt.close(fig)` |
| A comparison looks exaggerated | Inspect units, limits, baseline, and scale |

## Further reading

The official [quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html) explains the Figure/Axes interface. The [Figure API](https://matplotlib.org/stable/api/figure_api.html) documents figure layout and export. Use these references when extending the examples.

## Explain what the evidence supports

Write two sentences describing the weekly difference and one explaining what seven invented readings cannot establish. Support the description with the plotted values, not merely the visual impression.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: explain a latency regression that depends on traffic mix

A service release appears slower in an overall latency dashboard, yet engineers say each route improved. Reconstruct the report from 2,000 generated request events with a changed traffic mix, missing timeout measurements, three malformed values and 20 duplicate retries. Compare pooled and route-specific summaries, retain outcome counts, and control how much detail reaches the final chart. The data is a constructed operational scenario, not a measurement of a real service.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through a misleading pooled average

Suppose fast requests improve from 80 to 60 ms and slow requests from 800 to 600 ms. Before the release, traffic is mostly fast; afterward it is mostly slow. The weighted overall mean can rise even when both conditional means fall. Calculate both levels before drawing a conclusion. Timeouts are absent durations here, not zero milliseconds; report them separately. A causal claim about the release would need a stronger comparison design.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(72);p2_records=[]
for p2_period,p2_route,p2_count,p2_mean in [('before','fast',900,80),('before','slow',100,800),('after','fast',100,60),('after','slow',900,600)]:
    for p2_value in p2_rng.lognormal(np.log(p2_mean)-0.5*0.15**2,0.15,p2_count):
        p2_records.append({'request_id':len(p2_records),'period':p2_period,'route':p2_route,'latency_ms':str(round(p2_value,2)),'status':'ok'})
p2_raw=pd.DataFrame(p2_records)
p2_raw.loc[::97,'latency_ms']='';p2_raw.loc[::97,'status']='timeout'
p2_raw.loc[[11,12,13],'latency_ms']='broken'
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-20:]],ignore_index=True)
p2_unique=p2_raw.drop_duplicates().copy()
p2_unique['latency']=pd.to_numeric(p2_unique.latency_ms,errors='coerce')
p2_bad=(p2_unique.status=='ok')&p2_unique.latency.isna()
p2_observed=p2_unique[(p2_unique.status=='ok')&~p2_bad].copy()
p2_summary=p2_observed.groupby(['period','route']).latency.agg(count='size',median='median',mean='mean',p95=lambda x:x.quantile(.95)).reset_index()
p2_means=p2_summary.pivot(index='route',columns='period',values='mean')
assert (p2_means['after']<p2_means['before']).all()
p2_pooled=p2_observed.groupby('period').latency.mean()
assert p2_pooled['after']>p2_pooled['before']
assert len(p2_unique)==2000 and p2_bad.sum()==3
print({'raw':len(p2_raw),'unique':len(p2_unique),'timeouts':int((p2_unique.status=='timeout').sum()),'malformed_ok_rows':int(p2_bad.sum())})
display(p2_summary)

### Work through a report that preserves the denominator

Use one panel for observed mean latency by route and period, and another for the number of requests with each outcome. The first panel answers “how slow were measured successes?”; the second reveals what was excluded. Avoid a single pooled bar whose height mixes performance with changing traffic proportions.

For diagnostic scatter, cap points per group rather than plotting every event. Sampling can hide rare tail events, so compute p95 and counts on the full accepted population before sampling.

In [ ]:
import matplotlib.pyplot as plt
p2_fig,p2_axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
p2_means[['before','after']].plot.bar(ax=p2_axes[0],rot=0)
p2_axes[0].set(ylabel='Mean observed success latency (ms)',title='Each route improved')
p2_outcomes=p2_unique.assign(outcome=np.where(p2_bad,'malformed',p2_unique.status)).groupby(['period','outcome']).size().unstack(fill_value=0)
p2_outcomes.plot.bar(stacked=True,ax=p2_axes[1],rot=0)
p2_axes[1].set(ylabel='Unique requests',title='What the latency panel excludes')
p2_fig.savefig(p2_out/'matplotlib_release_report.png',dpi=140);plt.show()
p2_sample=p2_observed.groupby(['period','route'],group_keys=False).apply(lambda g:g.sample(min(100,len(g)),random_state=7),include_groups=False)
assert len(p2_sample)<=400
p2_summary.to_csv(p2_out/'matplotlib_release_summary.csv',index=False)
print({'pooled_means_ms':p2_pooled.to_dict(),'scatter_budget':400,'sampled_rows':len(p2_sample)})

### Interpret what changed

The apparent aggregate regression is consistent with a shift toward slow-route traffic. Both route-specific means decreased, but the report does not prove a treatment effect or account for unmeasured timeouts. The bars start at zero so their lengths remain interpretable; the units specify milliseconds rather than an unlabeled “latency.”

**Try:** replace the mean with p95. **Worked reasoning:** p95 is not the weighted average of subgroup p95 values. Compute it from the intended event population. A sampled scatter is a navigation aid, not the source for tail-percentile calculations. Large SVGs may be reduced with rasterized point layers while keeping text/vector axes readable.